# HPT prototype on an A100 — UI and the 600-case eval from one vLLM instance

Google Colab → **Runtime → Change runtime type → A100 GPU**.

vLLM is never served here. `vllm.LLM(...)` loads the weights into this notebook's
own process and returns an object; generation is a method call, no port, no
`vllm serve`. The single instance built in cell 3 is handed to both the UI and the
eval, so the card holds one copy of the weights.

Order matters: installs → data → model → UI → eval.

## 1. Code and dependencies

In [ ]:
%cd /content
import os
if not os.path.isdir("/content/HPT_Prototype"):
    !git clone https://github.com/adi274903/HPT_Prototype /content/HPT_Prototype
%cd /content/HPT_Prototype

!pip install -q --upgrade pip
!pip install -q vllm duckdb sentence-transformers qdrant-client pandas
!git log --oneline -1          # expect 9fa7348 or later

If `import vllm` fails afterwards: **Runtime → Restart session**, then re-run
from cell 2 (vLLM needs a clean CUDA context; the pip install stays cached).

## 2. Drive, Qdrant snapshot, MRF

In [ ]:
import colab_run
colab_run.mount_drive()                 # /content/drive
qdrant_path = colab_run.stage_db()      # cp + tar -xzf -> /content/New_PT_DB
print(colab_run.default_paths())

## 3. One vLLM instance, shared by the app and the eval

In [ ]:
from vllm import LLM
from pt_healthcare.backends import VLLMPipe

SHARED = LLM(
    model="google/medgemma-1.5-4b-it",
    dtype="bfloat16",
    max_model_len=8192,
    gpu_memory_utilization=0.85,   # 40 GB card; leaves room for the embedder
)

gateway  = VLLMPipe("google/medgemma-1.5-4b-it", llm=SHARED)   # the app's generator
pipeline = colab_run.setup(mount=False, stage=False, qdrant_path=qdrant_path, pipe=gateway)
print(pipeline.run("What might a diagnostic mammogram cost at UPMC Presbyterian?")["answer"])

MedGemma 1.5 under vLLM is the one piece I could not verify — no GPU here.
If this cell throws on model load, the known-good fallback is the transformers
path; every later cell still works, just slower:

```python
pipeline = colab_run.setup(mount=False, stage=False, qdrant_path=qdrant_path)
```

## 4. Serve the UI

In [ ]:
ui = colab_run.serve_ui(pipeline=pipeline, page="patient", tunnel="cloudflare")
print(ui.pages)        # {'patient': 'https://<id>.trycloudflare.com/', 'dashboard': '.../pipeline'}
# ui.stop()  when done — the tunnel is public and unauthenticated

## 5. The 600-case eval, on the same weights

In [ ]:
import os
if not os.path.exists(".cache/mrf_eval.duckdb"):     # once, ~8 min over the 2.3 GB CSV
    !python scripts/prepare_eval_mrf.py --mrf "/content/drive/MyDrive/everyUPMCmrf_clean.csv" --out .cache/mrf_eval.duckdb

In [ ]:
from pt_healthcare.eval_runner import VLLMInProcessClient, evaluate, format_report

report = evaluate(client=VLLMInProcessClient(llm=SHARED))   # all 600, resumes where it stopped
print(format_report(report))

report["codes"]["dice_mrf"], report["rows"]["dice_raw"], report["ner"]["hospital"]["f1"]

The eval and the UI share the GPU, so the UI stalls while eval generations run.
Start the eval when nobody is clicking, or run it before cell 4.

`evaluate()` appends every finished case to `eval_run.jsonl` and skips what is
already there — a dropped runtime costs the cases in flight, not the run. Pass
`fresh=True` to start over, `per_subtype=1` for a ten-case smoke test.

Ten-case smoke first, if you want a fast signal:

```python
report = evaluate(per_subtype=1, client=VLLMInProcessClient(llm=SHARED))
```